In [1]:
from qiime2 import Metadata
from urllib import request


In [2]:

# url = 'https://moving-pictures-tutorial.readthedocs.io/en/2026.4/data/moving-pictures/sample-metadata.tsv'
# fn = 'sample-metadata.tsv'
# request.urlretrieve(url, fn)
sample_metadata_md = Metadata.load("./input/sample-metadata.tsv")


In [ ]:
sample_metadata_md


Metadata
--------
34 IDs x 8 columns
barcode-sequence:            ColumnProperties(type='categorical', missing_scheme='blank')
body-site:                   ColumnProperties(type='categorical', missing_scheme='blank')
year:                        ColumnProperties(type='numeric', missing_scheme='blank')
month:                       ColumnProperties(type='numeric', missing_scheme='blank')
day:                         ColumnProperties(type='numeric', missing_scheme='blank')
subject:                     ColumnProperties(type='categorical', missing_scheme='blank')
reported-antibiotic-usage:   ColumnProperties(type='categorical', missing_scheme='blank')
days-since-experiment-start: ColumnProperties(type='numeric', missing_scheme='blank')

Call to_dataframe() for a tabular representation.

In [4]:
import rachis.plugins.metadata.actions as metadata_actions

sample_metadata_viz_viz, = metadata_actions.tabulate(
    input=sample_metadata_md,
)


In [ ]:
import zipfile

# url = 'https://moving-pictures-tutorial.readthedocs.io/en/2026.4/data/moving-pictures/emp-single-end-sequences.zip'
fn = './input/emp-single-end-sequences.zip'

with zipfile.ZipFile(fn) as zf:
    zf.extractall('emp-single-end-sequences')


In [ ]:
from rachis import Artifact

emp_single_end_sequences = Artifact.import_data(
    'EMPSingleEndSequences',
    'emp-single-end-sequences',
)


In [ ]:
emp_single_end_sequences


<artifact: EMPSingleEndSequences uuid: b8f5807b-2f96-4676-a50a-0f3f985a1500>

In [ ]:
import rachis.plugins.demux.actions as demux_actions

barcode_sequence_mdc = sample_metadata_md.get_column('barcode-sequence')
demux, demux_details = demux_actions.emp_single(
    seqs=emp_single_end_sequences,
    barcodes=barcode_sequence_mdc,
)


In [ ]:
demux_viz, = demux_actions.summarize(
    data=demux,
)


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:182: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  context['result_data'] = pd.concat([context['result_data'], df])
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_demux/_summarize/_visualizer.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['152 nts' '152 nts' '152 nts' '152 nts' '152 nts' '152 nts' '152 nts']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  stats[stats.index != 'count'] = \


<Figure size 640x480 with 0 Axes>

#### DADA2


In [ ]:
import rachis.plugins.dada2.actions as dada2_actions

table, rep_seqs, denoising_stats, base_transition_stats = dada2_actions.denoise_single(
    demultiplexed_seqs=demux,
    trim_left=0,
    trunc_len=120,
)


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: run_dada.R --input_directory /tmp/qiime2/root/data/a7e7a5a2-919c-4264-a4fe-4e7845d09d69/data --output_path /tmp/tmp8dgf0wxz/output.tsv.biom --output_track /tmp/tmp8dgf0wxz/track.tsv --output_err_track /tmp/tmp8dgf0wxz/err_track.tsv --filtered_directory /tmp/tmp8dgf0wxz --truncation_length 120 --trim_left 0 --max_expected_errors 2.0 --truncation_quality_score 2 --max_length Inf --pooling_method independent --chimera_method consensus --min_parental_fold 1.0 --allow_one_off False --num_threads 1 --learn_min_reads 1000000 --homopolymer_gap_penalty NULL --band_size 16

R version 4.5.2 (2025-10-31) 


Loading required package: Rcpp


DADA2: 1.38.0 / Rcpp: 1.1.1 / RcppParallel: 5.1.11.2 
2) Filtering ..................................
3) Learning Error Rates
19539480 total bases in 162829 reads from 34 samples will be used for learning the error rates.
4) Denoise samples 
..................................
5) Remove chimeras (method = consensus)
6) Report read numbers through the pipeline
7) Write output


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  series[missing.index] = missing


In [ ]:
from rachis import Metadata

stats_dada2_md_md = denoising_stats.view(Metadata)
denoising_stats_viz, = metadata_actions.tabulate(
    input=stats_dada2_md_md,
)


#### Deblur


In [ ]:
import rachis.plugins.quality_filter.actions as quality_filter_actions

demux_filtered, demux_filter_stats = quality_filter_actions.q_score(
    demux=demux,
)


In [13]:
import rachis.plugins.deblur.actions as deblur_actions

table_deblur, rep_seqs_deblur, deblur_stats = deblur_actions.denoise_16S(
    demultiplexed_seqs=demux_filtered,
    trim_length=120,
    sample_stats=True,
)


/opt/conda/envs/rachis-qiime2-2026.4/bin/deblur:534: DeprecationWarning: The 'warn' method is deprecated, use 'warning' instead
  logger.warn('deblur version %s workflow started on %s' %
/opt/conda/envs/rachis-qiime2-2026.4/bin/deblur:536: DeprecationWarning: The 'warn' method is deprecated, use 'warning' instead
  logger.warn('parameters: %s' % locals())


In [16]:
filter_stats_md = demux_filter_stats.view(Metadata)
demux_filter_stats_viz, = metadata_actions.tabulate(
    input=filter_stats_md,
)
deblur_stats_viz, = deblur_actions.visualize_stats(
    deblur_stats=deblur_stats,
)


In [17]:
table = table_deblur
rep_seqs = rep_seqs_deblur


In [18]:
import rachis.plugins.feature_table.actions as feature_table_actions

feature_frequencies, sample_frequencies, table_viz = feature_table_actions.summarize(
    table=table,
    metadata=sample_metadata_md,
)
rep_seqs_viz, = feature_table_actions.tabulate_seqs(
    data=rep_seqs,
)


In [ ]:
import rachis.plugins.phylogeny.actions as phylogeny_actions

aligned_rep_seqs, masked_aligned_rep_seqs, unrooted_tree, rooted_tree = phylogeny_actions.align_to_tree_mafft_fasttree(
    sequences=rep_seqs,
)


In [ ]:
import rachis.plugins.diversity.actions as diversity_actions

action_results = diversity_actions.core_metrics_phylogenetic(
    phylogeny=rooted_tree,
    table=table,
    sampling_depth=1103,
    metadata=sample_metadata_md,
)
rarefied_table = action_results.rarefied_table
faith_pd_vector = action_results.faith_pd_vector
observed_features_vector = action_results.observed_features_vector
shannon_vector = action_results.shannon_vector
evenness_vector = action_results.evenness_vector
unweighted_unifrac_distance_matrix = action_results.unweighted_unifrac_distance_matrix
weighted_unifrac_distance_matrix = action_results.weighted_unifrac_distance_matrix
jaccard_distance_matrix = action_results.jaccard_distance_matrix
bray_curtis_distance_matrix = action_results.bray_curtis_distance_matrix
unweighted_unifrac_pcoa_results = action_results.unweighted_unifrac_pcoa_results
weighted_unifrac_pcoa_results = action_results.weighted_unifrac_pcoa_results
jaccard_pcoa_results = action_results.jaccard_pcoa_results
bray_curtis_pcoa_results = action_results.bray_curtis_pcoa_results
unweighted_unifrac_emperor_viz = action_results.unweighted_unifrac_emperor
weighted_unifrac_emperor_viz = action_results.weighted_unifrac_emperor
jaccard_emperor_viz = action_results.jaccard_emperor
bray_curtis_emperor_viz = action_results.bray_curtis_emperor


In [21]:
faith_pd_group_significance_viz, = diversity_actions.alpha_group_significance(
    alpha_diversity=faith_pd_vector,
    metadata=sample_metadata_md,
)
evenness_group_significance_viz, = diversity_actions.alpha_group_significance(
    alpha_diversity=evenness_vector,
    metadata=sample_metadata_md,
)


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')


In [22]:
body_site_mdc = sample_metadata_md.get_column('body-site')
unweighted_unifrac_body_site_group_significance_viz, = diversity_actions.beta_group_significance(
    distance_matrix=unweighted_unifrac_distance_matrix,
    metadata=body_site_mdc,
    pairwise=True,
)
subject_mdc = sample_metadata_md.get_column('subject')
unweighted_unifrac_subject_group_significance_viz, = diversity_actions.beta_group_significance(
    distance_matrix=unweighted_unifrac_distance_matrix,
    metadata=subject_mdc,
    pairwise=True,
)


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_diversity/_beta/_visualizer.py:174: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pairs_summary = pd.concat([pairs_summary, group_pairs_summary])
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_diversity/_beta/_visualizer.py:179: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(x_ticklabels, rotation=90)
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_diversity/_beta/_visualizer.py:179: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(

<Figure size 640x480 with 0 Axes>

In [23]:
import rachis.plugins.emperor.actions as emperor_actions

unweighted_unifrac_emperor_days_since_experiment_start_viz, = emperor_actions.plot(
    pcoa=unweighted_unifrac_pcoa_results,
    metadata=sample_metadata_md,
    custom_axes=['days-since-experiment-start'],
)
bray_curtis_emperor_days_since_experiment_start_viz, = emperor_actions.plot(
    pcoa=bray_curtis_pcoa_results,
    metadata=sample_metadata_md,
    custom_axes=['days-since-experiment-start'],
)


In [24]:
alpha_rarefaction_viz, = diversity_actions.alpha_rarefaction(
    table=table,
    phylogeny=rooted_tree,
    max_depth=4000,
    metadata=sample_metadata_md,
)


Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

faithpd -i /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-3p7oc40k -t /tmp/qiime2/root/data/ab3ed559-e95a-4f4c-ba88-d45180797a73/data/tree.nwk -o /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-98vumzpd



/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].app

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

faithpd -i /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-7lfb5ro3 -t /tmp/qiime2/root/data/ab3ed559-e95a-4f4c-ba88-d45180797a73/data/tree.nwk -o /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-miv1ydbw



/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].app

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

faithpd -i /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-67x8nawk -t /tmp/qiime2/root/data/ab3ed559-e95a-4f4c-ba88-d45180797a73/data/tree.nwk -o /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-k3n9b5sa



...

/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].app

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

faithpd -i /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-m2atsvfo -t /tmp/qiime2/root/data/ab3ed559-e95a-4f4c-ba88-d45180797a73/data/tree.nwk -o /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-7uqfbt5o



/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].app

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

faithpd -i /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-3fvt1ng5 -t /tmp/qiime2/root/data/ab3ed559-e95a-4f4c-ba88-d45180797a73/data/tree.nwk -o /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-e2zqhikd



/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].apply(pd.to_numeric, errors='ignore')
/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_types/sample_data/_deferred_setup/_transformers.py:28: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[cols] = df[cols].app

In [28]:
from qiime2 import Artifact

# url = 'https://moving-pictures-tutorial.readthedocs.io/en/2026.4/data/moving-pictures/reference-sequences.qza'
# fn = 'reference-sequences.qza'
# request.urlretrieve(url, fn)
reference_sequences = Artifact.load("./input/reference-sequences.qza")


In [29]:
# url = 'https://moving-pictures-tutorial.readthedocs.io/en/2026.4/data/moving-pictures/reference-taxonomy.qza'
# fn = 'reference-taxonomy.qza'
# request.urlretrieve(url, fn)
reference_taxonomy = Artifact.load("./input/reference-taxonomy.qza")


In [30]:
import rachis.plugins.feature_classifier.actions as feature_classifier_actions

suboptimal_16S_rRNA_classifier, = feature_classifier_actions.fit_classifier_naive_bayes(
    reference_reads=reference_sequences,
    reference_taxonomy=reference_taxonomy,
)


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_feature_classifier/classifier.py:106: UserWarning: The TaxonomicClassifier artifact that results from this method was trained using scikit-learn version 1.7.1. It cannot be used with other versions of scikit-learn. (While the classifier may complete successfully, the results will be unreliable.)
  warnings.warn(warning, UserWarning)


In [32]:
taxonomy, = feature_classifier_actions.classify_sklearn(
    classifier=suboptimal_16S_rRNA_classifier,
    reads=rep_seqs,
)
taxonomy_as_md_md = taxonomy.view(Metadata)
taxonomy_viz, = metadata_actions.tabulate(
    input=taxonomy_as_md_md,
)


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_feature_classifier/_taxonomic_classifier.py:71: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dirname)


In [33]:
import rachis.plugins.taxa.actions as taxa_actions

taxa_bar_plots_viz, = taxa_actions.barplot(
    table=table,
    taxonomy=taxonomy,
    metadata=sample_metadata_md,
)


In [34]:
gut_table, = feature_table_actions.filter_samples(
    table=table,
    metadata=sample_metadata_md,
    where='[body-site]="gut"',
)


In [35]:
import rachis.plugins.composition.actions as composition_actions

ancombc_subject, = composition_actions.ancombc(
    table=gut_table,
    metadata=sample_metadata_md,
    formula='subject',
)
da_barplot_subject_viz, = composition_actions.da_barplot(
    data=ancombc_subject,
    significance_threshold=0.001,
)


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_composition/_ancombc.py:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  term_alpha_value = (metadata.get_column(term)


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: run_ancombc.R --inp_abundances_path /tmp/tmpcq6gaa30/input.biom.tsv --inp_metadata_path /tmp/tmpcq6gaa30/input.map.txt --md_column_types {"barcode-sequence": "categorical", "body-site": "categorical", "year": "numeric", "month": "numeric", "day": "numeric", "subject": "categorical", "reported-antibiotic-usage": "categorical", "days-since-experiment-start": "numeric"} --formula subject --p_adj_method holm --prv_cut 0.1 --lib_cut 0 --reference_levels ['subject::subject-1'] --tol 1e-05 --max_iter 100 --conserve False --alpha 0.05 --output_loaf /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-lcg57sle



── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘jsonlite’

The following object is masked from ‘package:purrr’:

    flatten



R version 4.5.2 (2025-10-31) 


New names:
• `` -> `...1`
'ancombc' has been fully evolved to 'ancombc2'. 
Explore the enhanced capabilities of our refined method!
Checking the input data type ...
The input data is of type: phyloseq
PASS
Checking the sample metadata ...
The specified variables in the formula: subject
The available variables in the sample metadata: barcode.sequence, body.site, year, month, day, subject, reported.antibiotic.usage, days.since.experiment.start
PASS
Checking other arguments ...
PASS
Obtaining initial estimates ...
Estimating sample-specific biases ...
Loading required package: foreach

Attaching package: ‘foreach’

The following objects are masked from ‘package:purrr’:

    accumulate, when

Loading required package: rngtools
ANCOM-BC primary results ...


In [36]:
gut_table_l6, = taxa_actions.collapse(
    table=gut_table,
    taxonomy=taxonomy,
    level=6,
)
l6_ancombc_subject, = composition_actions.ancombc(
    table=gut_table_l6,
    metadata=sample_metadata_md,
    formula='subject',
)
l6_da_barplot_subject_viz, = composition_actions.da_barplot(
    data=l6_ancombc_subject,
    significance_threshold=0.001,
)


/opt/conda/envs/rachis-qiime2-2026.4/lib/python3.12/site-packages/q2_composition/_ancombc.py:64: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  term_alpha_value = (metadata.get_column(term)


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: run_ancombc.R --inp_abundances_path /tmp/tmp1galotrj/input.biom.tsv --inp_metadata_path /tmp/tmp1galotrj/input.map.txt --md_column_types {"barcode-sequence": "categorical", "body-site": "categorical", "year": "numeric", "month": "numeric", "day": "numeric", "subject": "categorical", "reported-antibiotic-usage": "categorical", "days-since-experiment-start": "numeric"} --formula subject --p_adj_method holm --prv_cut 0.1 --lib_cut 0 --reference_levels ['subject::subject-1'] --tol 1e-05 --max_iter 100 --conserve False --alpha 0.05 --output_loaf /tmp/qiime2/root/processes/69-1784582723.85@root/tmp/rachis-OutPath-3jq9imb8



── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘jsonlite’

The following object is masked from ‘package:purrr’:

    flatten



R version 4.5.2 (2025-10-31) 


New names:
• `` -> `...1`
'ancombc' has been fully evolved to 'ancombc2'. 
Explore the enhanced capabilities of our refined method!
Checking the input data type ...
The input data is of type: phyloseq
PASS
Checking the sample metadata ...
The specified variables in the formula: subject
The available variables in the sample metadata: barcode.sequence, body.site, year, month, day, subject, reported.antibiotic.usage, days.since.experiment.start
PASS
Checking other arguments ...
PASS
Obtaining initial estimates ...
Estimating sample-specific biases ...
Loading required package: foreach

Attaching package: ‘foreach’

The following objects are masked from ‘package:purrr’:

    accumulate, when

Loading required package: rngtools
ANCOM-BC primary results ...
